<div class="alert alert-info"> 
    
# IX. Exercise: Putting It All Together

The power of Xarray is most apparent when working with data that has three or more dimensions. We saw many examples of that in the Xarray lesson. However, in this exercise you'll work with the same 1D and 2D data that was used in the NumPy exercise and do much of the same analysis. The purpose of this is to give you more practice with file input/output, creating simpler Xarray data structures, and clearly demonstrating differences between analysis with unlabeled (NumPy) vs labeled (Xarray) array data structures.

This exercise is in a separate file from the main lesson so that you can more easily reference the content in other notebooks if you need to, by having multiple files open side-by-side in Jupyter Lab.

In this exercise you will use Xarray (as well as NumPy, Pandas, and Matplotlib) to read .txt and .nc files, create Xarray DataArrays, execute calculations, and visualize results. The data files contain the Nino3.4 Index as well as monthly precipitation anomalies at 10 different US cities. The overarching task will be to get this data into Xarray DataArrays, calculate the mean precipitation anomaly at each city during El Nina and La Nina months, and visualize your calculations.
</div>

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

<div class="alert alert-info"> 

## 1) Intro to the El Nino Southern Oscillation (ENSO)

**The information in this section is an exact copy from the NumPy lesson. If you remember this content, feel free to skip to the next section. The content is repeated here for your convenience in case ENSO is a new concept for you or if it has been a while since you completed the NumPy lesson.**

Start by watching this short video about the El Nino Southern Oscillation climate phenomenon and how it affects precipitation and temperature in the United States.

<video controls width="560" src="https://oceanservice.noaa.gov/facts/elninolanina/otkn_721_elninolanina_lg.mp4"></video>

As you saw in the video, ENSO is a feature of Earth's natural climate variability that impacts weather globally. It has 3 phases: El Nino (warm tropical Pacific), La Nina (cool tropical Pacific), and neutral (normal tropical Pacific). Over North America, the phase of ENSO modulates the location of the jet stream and the storm systems that travel along it. This results in areas of wetter/drier and cooler/warmer weather during El Nino and La Nina conditions. Weather impacts from El Nino and La Nina are generally strongest in winter months over North America. The strongest ENSO influence on US precipitation is seen in the southern United States from southern California to Florida. 

Scientists have developed a few different *indexes* for tracking the phase and strength of ENSO over time. The ENSO index we will use in this exercise is called the Nino3.4 Index. This index is created from monthly sea surface temperature anomalies (departure from the long term mean) from a region in the tropical Pacific ocean known as the Nino3.4 region. 

For the purposes of this exercise, we will select months with El Nino and La Nina using the following criteria:
- El Nino conditions - months when the Index is greater than or equal to +1
- La Nina conditions - months when the Index is less than or equal to -1
  
The way research scientists identify El Nino and La Nina events from the Nino3.4 Index is little bit more complex. We won't get into the details here though since this exercise is focused on NumPy practice, not scientific rigor.
</div>

<div class="alert alert-info"> 

## 2) Read ENSO Data, Convert to DataArray, and Visualize

This time around we'll use Pandas and NumPy to read the data and convert to Xarray. 

Read the file at ```data/NOAA_ClimateIndex/Nino34ClimateIndex.txt``` into a Pandas DataFrame called ```enso``` and render a preview of the DataFrame to the screen. Make sure to use function parameters to skip any header and footer rows in the data file and assign NaN to the missing data value of -99.99. 

**Hint:** We've already done this in the NumPy lesson.
</div>

In [ ]:
# add your code here
enso = pd.read_csv(r'../data/NOAA_ClimateIndex/Nino34ClimateIndex.txt',
                   header=None,
                   skiprows=1,
                   skipfooter=3,
                   na_values=-99.99,
                   delimiter=r'\s+',
                   names=np.arange(1,13),
                   engine='python')
enso

<div class="alert alert-info"> 

Overwrite the ```enso``` DataFrame, dropping rows that contain NaN using a Pandas DataFrame function. Then, use a Pandas function to count the number of missing values remaining in the DataFrame, verifying that the answer is zero. Print a preview of the resulting ```enso``` DataFrame.

In [ ]:
# add your code here
enso = enso.dropna(how='any')
# enso.dropna(how='any',inplace=True)

print(enso.isna().sum())#.sum())
enso

<div class="alert alert-info"> 

Create a Pandas DatetimeIndex called ```time``` of monthly datetimes for all the years remaining in the DataFrame. The day in each datetime should be the first day of the month. 


In [ ]:
# add your code here
time = pd.date_range('1950-01','2024-12',freq='MS')
time

<div class="alert alert-info"> 

Convert the ```enso``` DataFrame to a 1D NumPy array called ```enso_np``` where the values of the Nino3.4 Index are sequential in time from 1950 to 2024. 

**Hint:** Reference the following sections in the NumPy lesson for the functions you will need:
- Converting between Pandas and NumPy Data Structures
- Removing and Collapsing Dimensions
</div>

In [ ]:
# add your code here
enso_np = enso.to_numpy().flatten()

<div class="alert alert-info"> 

Check your work by:
- using an ```np.testing``` assert function to test if the first 12 values of the ```enso_np``` array are equal to the first row of ```enso``` DataFrame. **Hint:** you can convert data from a Pandas to a NumPy data structure with the Pandas function ```.to_numpy()```.
- use Python ```assert()``` to test if the length of the ```enso_np``` array is equal to the length of the ```time``` DatetimeIndex

In [ ]:
# add your code here
np.testing.assert_array_equal(enso_np[0:12],enso.loc[1950].to_numpy())
np.testing.assert_array_equal(enso_np[0:12],enso.iloc[0].to_numpy()) # alternative

assert len(enso_np)==len(time),'length of enso_arr and time do not match'

<div class="alert alert-info"> 

Convert the ```enso_np``` array to a 1D Xarray DataArray called ```enso_xr``` and include the following metadata/labels:
- dimension name is time
- coordinate values equal to the ```time``` DatetimeIndex
- time attributes ```{'standard_name':'time', 'long_name':'time, in monthly increments`}```
- variable attributes ```{'standard_name':'nino34', 'units':'degree_C'}```
</div>

In [ ]:
# add your code here

enso_xr = xr.DataArray(enso_np, coords = {'time':('time',time)})

enso_xr.time.attrs = {'standard_name':'time', 'long_name':'time, in monthly increments'}
enso_xr.attrs = {'standard_name':'nino34', 'units':'degree_C'}

enso_xr

<div class="alert alert-info"> 

Just for extra practice, count how many total NaNs there are in ```enso_xr``` using NumPy or Xarray functions. 

**Hint:** We covered the NumPy function to count NaNs but we didn't cover the similar Xarray function for this: ```.isnull()```. See if you can get the same result using both methods.

</div>

In [ ]:
# add your code here
print(np.isnan(enso_xr).sum())
print(enso_xr.isnull().sum())

<div class="alert alert-info"> 

Finally, use Xarray ```.plot()``` to plot the Nino3.4 Index timeseries (```enso_xr```). Include the parameter ```figsize=(10,2)``` and add a grey dashed reference line at y=0.

</div>

In [ ]:
# add your code here
enso_xr.plot(figsize=(10,2))
plt.axhline(0,color='grey',ls='dashed')
plt.show()

**Interpreting the figure above:** The further away from zero the index is (in either direction), the stronger the ENSO event. The more positive the index is, the stronger the EL Nino phenomenon. The more negative the index is, the stronger the La Nina phenomenon. Neutral conditions occur when the index is closer to zero. 

<div class="alert alert-info"> 

## 3) Read Netcdf file to Xarray Dataset and Practice Manipulating Xarray Data Structures

The monthly precipitation anomalies from the nclimgrid data for the same 10 cities we worked with in the NumPy exercise are provided in the netcdf file ```data/nclimgrid/pr_anom_nclimgrid_monthly_10cities.nc```. 

Read the file into an Xarray Dataset called ```ds``` and print the ```ds``` metadata to the screen.
</div>

In [ ]:
# add your code here

ds = xr.open_dataset('../data/nclimgrid/pr_anom_nclimgrid_monthly_10cities.nc')
ds

<div class="alert alert-info"> 

Use an ```xr.testing``` assert statement to check if the times in ```ds``` match the times in ```enso_xr```.
</div>

In [ ]:
# add your code here
xr.testing.assert_equal(ds.time,enso_xr.time)

<div class="alert alert-info"> 

Overwrite ```ds``` so that the times match ```enso_xr```. **Hint:** You can accomplish this either by slicing in time or using advanced indexing. 

Afterward, paste a copy of the ```xr.testing``` assert function from above and run it to check again for a match between all times.
</div>

In [ ]:
# add your code here
ds = ds.sel(time=enso_xr.time)
# ds = ds.sel(time=slice(enso_xr.time[0],enso_xr.time[-1]))

xr.testing.assert_equal(ds.time,enso_xr.time)

<div class="alert alert-info"> 

Save the data variable "pr_anom" in the Dataset ```ds``` to a new Xarray DataArray object called ```pr```.
</div>

In [ ]:
# add your code here
pr = ds.pr_anom
pr

<div class="alert alert-info"> 

Using the ```pr``` DataArray, programmatically show the longitude of the Tucson location using Xarray functions.
</div>

In [ ]:
# add your code here
# pr.lon.sel(location='Tucson').item()#.data
pr.sel(location='Tucson').lon.item()#.data

<div class="alert alert-info"> 
    
Count how many NaNs there are in the ```pr``` array. This can be accomplished with Xarray or NumPy functions, the choice is yours.
</div>

In [ ]:
# add your code here
pr.isnull().sum()
np.isnan(pr).sum()

<div class="alert alert-info"> 
    
Convert ```pr``` from units of mm to units of inches (this is an example of broadcasting). 
</div>

In [ ]:
# add your code here
pr = pr/25.4
pr

<div class="alert alert-info"> 

Find the most positive and most negative value of ```pr``` at each location.

In [ ]:
# add your code here
pr.max('time')

In [ ]:
# add your code here
pr.min('time')

<div class="alert alert-info"> 

## 4) Use Advanced Indexing to Find Mean Precipitation Anomalies during El Nino and La Nina Months

First, using a comparison operator, create True/False masks to identify El Nino and La Nina months in the Nino3.4 Index.
- Create a variable called ```nino_mask``` where ```enso_xr``` meets or exceeds a value of +1.0
- Create a variable called ```nina_mask``` where ```enso_xr``` meets or exceeds a value of -1.0

Your ```nino_mask``` and ```nina_mask``` objects should be Xarray DataArrays.


In [ ]:
# add your code here
nino_mask = (enso_xr>=1)
nina_mask = (enso_xr<=-1)

nino_mask

<div class="alert alert-info"> 

How many El Nino months are there in ```nino_mask``` and how many La Nina months in ```nina_mask```?

In [ ]:
# add your code here
nino_mask.sum(),nina_mask.sum()

<div class="alert alert-info"> 
    
Use an Xarray function to select the time values in ```nino_mask``` that correspond to the El Nino months. In other words, display the datetimes where the ```nino_mask``` is True.

In [ ]:
# add your code here
nino_mask.sel(time=nino_mask).time

<div class="alert alert-info"> 

Chain additional functions onto the line of code you just wrote to answer the question: How many times does each month of the year occur in the 82 El Nino months?

**Hint:** This is something we didn't cover in the main lesson. The [list of methods you can use with ```.groupby()``` from the Xarray API reference](https://docs.xarray.dev/en/stable/generated/xarray.core.groupby.DataArrayGroupBy.html#xarray.core.groupby.DataArrayGroupBy) may be helpful. 

In [ ]:
# add your code here
nino_mask.sel(time=nino_mask).time.groupby('time.month').count()

<div class="alert alert-info"> 

Using an Xarray function, select only the values of ```pr``` for each city that correspond to El Nino months. You don't need to save this as a separate variable. It's just practice for the next step.

**Hint:** Use ```nino_mask``` as an indexer. 

In [ ]:
# add your code here
pr.sel(time=nino_mask)

<div class="alert alert-info"> 

Use advanced indexing to find the mean of ```pr``` for each city during:
-  El Nino months (```nino_mask```), save to a variable called ```nino_mean```
-  La Nina months (```nina_mask```), save to a variable called ```nina_mean```

Print both DataArrays to the screen to verify that they are both of length 10.
</div>

In [ ]:
# add your code here
nino_mean = pr.sel(time=nino_mask).mean('time')
nina_mean = pr.sel(time=nina_mask).mean('time')

nino_mean

In [ ]:
nina_mean

<div class="alert alert-info"> 

## 5) Elementwise Math, Sorting, and Visualization

Subtract the mean pr anomaly during La Nina months (```nina_mean```) from the mean pr anomaly during El Nino months (```nino_mean```) for each city (elementwise math) and save the result to a variable called ```diff```.
</div>

In [ ]:
# add your code here
diff = nino_mean - nina_mean
diff

<div class="alert alert-info"> 

Answer the following questions programmatically by accessing the location coordinate values (the city names) in ```nino_mean```, ```nina_mean```, and ```diff```:
- Which city has the largest mean monthly precip anomaly (most positive value) during El Nino months?
- Which city has the largest mean monthly precip anomaly (most negative value) during La Nina months?
- Which city has the largest difference between mean monthly precip anomalies (most positive value) in El Nino vs La Nina months?

**Hint:** Use Python built-in ```max()``` and ```min()```.

</div>

In [ ]:
max(nino_mean).location

In [ ]:
# add your code here
print(max(nino_mean).location.item())
print(min(nina_mean).location.data)
print(max(diff).location.data)

<div class="alert alert-info"> 

Use matplotlib in combination with Xarray ```.plot()``` to plot your ```diff``` results in a figure with the following:
- x axis shows the city names,
- y axis shows the ```diff``` values,
- lineweight is zero (no line) and marker is a shape of your choice,
- include a y axis label with units,
- include a plot title,
- rotate your x axis tick labels (city names) with ```plt.xticks(rotation=300,ha='left')```

**Hint: Copy the plot you created in the NumPy Putting It All Together Exercise and modify it to plot ```diff``` with Xarray ```.plot()```.**

</div>

In [ ]:
# add your code here
fig = plt.figure(figsize=(8,2))
diff.plot(marker='o',lw=0)
plt.xticks(rotation=300,ha='left')
plt.ylabel('Precipitation (inches)')
plt.title('nino minus nina mean monthly precip anomaly difference')
plt.show()

<div class="alert alert-info"> 

Use an Xarray function to sort the ```diff``` values in ascending order and save the result to a new variable called ```sorted_diff```. 

</div>

In [ ]:
# add your code here to sort
sorted_diff = diff.sortby(diff)
sorted_diff

<div class="alert alert-info"> 

Now, recreate the plot above, but make it a bar chart using matplotlib.pyplot's [```plt.bar()```](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html) with the city names and ```sorted_diff``` values appearing in ascending order from left to right. 

**Hint: These instructions are the same as in the NumPy exercise. Copy the plot you created in that exercise and modify it to plot the Xarray DataArray ```sorted_diff```.**

In [ ]:
# add your code here to plot
fig = plt.figure(figsize=(8,3))
plt.bar(sorted_diff.location,sorted_diff)
plt.xticks(rotation=300,ha='left')
plt.ylabel('Precipitation (inches)')
plt.title('nino minus nina mean monthly precip anomaly difference')
plt.show()